# 어린이 보호구역 사고 발생여부 이진분류 모델 비교 파이프라인
- 데이터: seongnam_train.csv
- 타겟: 발생건수 > 0 → 1, 발생건수 == 0 → 0 (이진화)
- 피처: 나머지 스케일링된 컬럼 (StandardScaling 완료)
- 비교 모델: Logistic Regression, SVM, Random Forest, XGBoost, KNN

In [ ]:
# 라이브러리 설치 (없을 경우)
import subprocess, sys

required = ["pandas", "numpy", "scikit-learn", "xgboost", "matplotlib", "seaborn"]
for pkg in required:
    try:
        __import__(pkg.replace("-", "_").replace("scikit_learn", "sklearn"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print("라이브러리 준비 완료")

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    classification_report, confusion_matrix, roc_curve
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

OUTPUT_DIR = "/mnt/user-data/outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("임포트 완료")
print(f"출력 경로: {OUTPUT_DIR}")

## STEP 0 | 데이터 로드 및 준비

In [ ]:
print("=== STEP 0 | 데이터 로드 및 준비 ===")

# 1. 데이터 로드 (실행 위치에 따라 경로 자동 조정)
candidates = [
    # "../ML/data/seongnam_train.csv",
    "data/seongnam_train.csv",
    # "ML/data/seongnam_train.csv",
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), candidates[0])

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print(f"데이터 로드 완료: {df.shape[0]}행 x {df.shape[1]}열")
print("전체 컬럼:", list(df.columns))

# 2. 타겟 이진화
TARGET_COL = "발생건수"
y = (df[TARGET_COL] > 0).astype(int)

# 3. 피처 선택 (시설물명·위도·경도·발생건수 제외)
DROP_COLS = ["시설물명", "위도", "경도", TARGET_COL]
feature_cols = [c for c in df.columns if c not in DROP_COLS]
X = df[feature_cols].copy()

print(f"\n사용 피처 ({len(feature_cols)}개):", feature_cols)

# 4. 클래스 분포
print("\n[전체 클래스 분포]")
class_counts = y.value_counts().sort_index()
total = len(y)
for label, cnt in class_counts.items():
    print(f"  클래스 {label}: {cnt}개 ({cnt/total*100:.1f}%)")

print("\n>>> STEP 0 완료")

## STEP 1 | 데이터 분할

In [ ]:
print("=== STEP 1 | 데이터 분할 ===")

# Stratified split: train 70% / val 15% / test 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}개  Val: {len(X_val)}개  Test: {len(X_test)}개")

def print_dist(name, y_split):
    counts = y_split.value_counts().sort_index()
    n = len(y_split)
    parts = [f"클래스 {k}: {v}개 ({v/n*100:.1f}%)" for k, v in counts.items()]
    print(f"  [{name}] " + " | ".join(parts))

print("\n분할별 클래스 분포:")
print_dist("Train", y_train)
print_dist("Val  ", y_val)
print_dist("Test ", y_test)

print("\n>>> STEP 1 완료")

## STEP 2 | 모델 정의

In [ ]:
print("=== STEP 2 | 모델 정의 ===")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM":                 SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric="logloss"),
    "KNN":                 KNeighborsClassifier(n_neighbors=5)
}

for name, model in models.items():
    print(f"  {name}: {model.__class__.__name__}")

print("\n>>> STEP 2 완료")

## STEP 3 | 학습 및 Val 평가

In [ ]:
print("=== STEP 3 | 학습 및 Val 평가 ===")

results_val    = {}
trained_models = {}
val_proba      = {}

for name, model in models.items():
    print(f"\n--- {name} ---")
    try:
        model.fit(X_train, y_train)
        trained_models[name] = model

        y_pred  = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]
        val_proba[name] = y_proba

        acc      = accuracy_score(y_val, y_pred)
        macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
        auc_roc  = roc_auc_score(y_val, y_proba)
        prec_1   = precision_score(y_val, y_pred, pos_label=1, zero_division=0)
        rec_1    = recall_score(y_val, y_pred, pos_label=1, zero_division=0)
        f1_1     = f1_score(y_val, y_pred, pos_label=1, zero_division=0)

        results_val[name] = {
            "Accuracy":      round(acc, 4),
            "Macro_F1":      round(macro_f1, 4),
            "AUC_ROC":       round(auc_roc, 4),
            "Class1_Prec":   round(prec_1, 4),
            "Class1_Recall": round(rec_1, 4),
            "Class1_F1":     round(f1_1, 4)
        }

        print(f"  Accuracy={acc:.4f}  Macro F1={macro_f1:.4f}  AUC-ROC={auc_roc:.4f}")
        print(f"  클래스1  Precision={prec_1:.4f}  Recall={rec_1:.4f}  F1={f1_1:.4f}")
        print(classification_report(y_val, y_pred,
                                    target_names=["클래스0(미발생)", "클래스1(발생)"],
                                    zero_division=0))
    except Exception as e:
        print(f"  [ERROR] {name} 실패: {e}")

print("\n>>> STEP 3 완료")

## STEP 4 | Val 결과 비교표

In [ ]:
print("=== STEP 4 | Val 결과 비교표 출력 ===")

df_val = pd.DataFrame(results_val).T.reset_index()
df_val.columns = ["모델명", "Accuracy", "Macro_F1", "AUC_ROC",
                  "Class1_Prec", "Class1_Recall", "Class1_F1"]
df_val = df_val.sort_values("AUC_ROC", ascending=False).reset_index(drop=True)

print("\n[Val 성능 비교표 (AUC_ROC 내림차순)]")
print(df_val[["모델명", "Accuracy", "Macro_F1", "AUC_ROC", "Class1_F1"]].to_string(index=False))

best_model_name = df_val.iloc[0]["모델명"]
best_auc        = float(df_val.iloc[0]["AUC_ROC"])
print(f"\nval AUC-ROC 기준 최적 모델: {best_model_name} (AUC={best_auc:.3f})")

print("\n>>> STEP 4 완료")

## STEP 5 | 시각화

In [ ]:
print("=== STEP 5 | 시각화 ===")

import platform
system = platform.system()
if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

MODEL_NAMES = list(results_val.keys())
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

# ── Figure 1: 모델별 지표 비교 막대그래프 (2x2) ──
metrics_info = [
    ("Accuracy",  "Accuracy"),
    ("Macro_F1",  "Macro F1"),
    ("AUC_ROC",   "AUC-ROC"),
    ("Class1_F1", "클래스1 F1"),
]

fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle("모델별 성능 지표 비교 (Validation Set)", fontsize=15, fontweight="bold")

for ax, (mkey, mlabel) in zip(axes.flatten(), metrics_info):
    values = [results_val[m][mkey] for m in MODEL_NAMES]
    bars = ax.bar(MODEL_NAMES, values, color=COLORS, edgecolor="white")
    ax.set_title(mlabel, fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
save_path1 = os.path.join(OUTPUT_DIR, "model_comparison.png")
plt.savefig(save_path1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure 1 저장: {save_path1}")

# ── Figure 2: ROC Curve ──
fig2, ax2 = plt.subplots(figsize=(8, 6))
for (name, proba), color in zip(val_proba.items(), COLORS):
    try:
        fpr, tpr, _ = roc_curve(y_val, proba)
        ax2.plot(fpr, tpr, color=color, lw=2,
                 label=f"{name} (AUC={results_val[name]['AUC_ROC']:.3f})")
    except Exception as e:
        print(f"  ROC 오류 [{name}]: {e}")

ax2.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random (AUC=0.500)")
ax2.set_xlabel("False Positive Rate", fontsize=12)
ax2.set_ylabel("True Positive Rate", fontsize=12)
ax2.set_title("ROC Curve 비교 (Validation Set)", fontsize=14, fontweight="bold")
ax2.legend(loc="lower right", fontsize=10)
ax2.grid(alpha=0.3)
plt.tight_layout()
save_path2 = os.path.join(OUTPUT_DIR, "roc_curve.png")
plt.savefig(save_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure 2 저장: {save_path2}")

# ── Figure 3: Random Forest 피처 중요도 ──
try:
    rf_model    = trained_models["Random Forest"]
    importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()
    bar_colors  = ["#c44e52" if v == importances.max() else "#4c72b0" for v in importances.values]

    fig3, ax3 = plt.subplots(figsize=(8, max(5, len(feature_cols) * 0.45)))
    importances.plot(kind="barh", ax=ax3, color=bar_colors, edgecolor="white")
    ax3.set_xlabel("Feature Importance", fontsize=12)
    ax3.set_title("Random Forest 피처 중요도", fontsize=14, fontweight="bold")
    ax3.grid(axis="x", alpha=0.3)
    for patch, val in zip(ax3.patches, importances.values):
        ax3.text(patch.get_width() + 0.002, patch.get_y() + patch.get_height() / 2,
                 f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    save_path3 = os.path.join(OUTPUT_DIR, "feature_importance.png")
    plt.savefig(save_path3, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure 3 저장: {save_path3}")
except Exception as e:
    print(f"  [ERROR] 피처 중요도 실패: {e}")

# ── Figure 4: Confusion Matrix (5개 모델 1행) ──
try:
    n_models = len(trained_models)
    fig4, axes4 = plt.subplots(1, n_models, figsize=(4 * n_models, 4))
    if n_models == 1:
        axes4 = [axes4]

    for ax, (name, model) in zip(axes4, trained_models.items()):
        cm = confusion_matrix(y_val, model.predict(X_val))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["미발생(0)", "발생(1)"],
                    yticklabels=["미발생(0)", "발생(1)"],
                    cbar=False)
        ax.set_title(name, fontsize=10, fontweight="bold")
        ax.set_xlabel("예측값", fontsize=9)
        ax.set_ylabel("실제값", fontsize=9)

    fig4.suptitle("Confusion Matrix 비교 (Validation Set)",
                  fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    save_path4 = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
    plt.savefig(save_path4, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure 4 저장: {save_path4}")
except Exception as e:
    print(f"  [ERROR] Confusion Matrix 실패: {e}")

print("\n>>> STEP 5 완료")

## STEP 6 | 최적 모델 테스트셋 최종 평가

In [ ]:
print("=== STEP 6 | 최적 모델 테스트셋 최종 평가 ===")

results_test = {}

try:
    best_model = trained_models[best_model_name]
    print(f"최적 모델: {best_model_name}  (val AUC={best_auc:.3f})")

    y_test_pred  = best_model.predict(X_test)
    y_test_proba = best_model.predict_proba(X_test)[:, 1]

    test_acc      = accuracy_score(y_test, y_test_pred)
    test_macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    test_auc      = roc_auc_score(y_test, y_test_proba)
    test_f1_1     = f1_score(y_test, y_test_pred, pos_label=1, zero_division=0)

    results_test[best_model_name] = {
        "Accuracy":  round(test_acc, 4),
        "Macro_F1":  round(test_macro_f1, 4),
        "AUC_ROC":   round(test_auc, 4),
        "Class1_F1": round(test_f1_1, 4)
    }

    print("\n[Test Set 최종 지표]")
    print(f"  Accuracy  : {test_acc:.4f}")
    print(f"  Macro F1  : {test_macro_f1:.4f}")
    print(f"  AUC-ROC   : {test_auc:.4f}")
    print(f"  클래스1 F1: {test_f1_1:.4f}")
    print("\n[Classification Report - Test Set]")
    print(classification_report(y_test, y_test_pred,
                                target_names=["클래스0(미발생)", "클래스1(발생)"],
                                zero_division=0))

    # 실제값 vs 예측확률 산점도
    y_test_arr = y_test.values if hasattr(y_test, "values") else np.array(y_test)
    idx_0 = y_test_arr == 0
    idx_1 = y_test_arr == 1

    fig5, ax5 = plt.subplots(figsize=(9, 5))
    ax5.scatter(np.where(idx_0)[0], y_test_proba[idx_0],
                color="steelblue", alpha=0.7, label="실제 0 (미발생)", s=60)
    ax5.scatter(np.where(idx_1)[0], y_test_proba[idx_1],
                color="tomato", alpha=0.8, label="실제 1 (발생)", marker="^", s=70)
    ax5.axhline(0.5, color="gray", linestyle="--", linewidth=1.2, label="임계값 0.5")
    ax5.set_xlabel("샘플 인덱스", fontsize=12)
    ax5.set_ylabel("예측 확률 (클래스1)", fontsize=12)
    ax5.set_title(f"실제값 vs 예측확률  [{best_model_name} · Test Set]",
                  fontsize=13, fontweight="bold")
    ax5.legend(fontsize=10)
    ax5.grid(alpha=0.3)
    plt.tight_layout()
    save_path5 = os.path.join(OUTPUT_DIR, "final_scatter.png")
    plt.savefig(save_path5, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"산점도 저장: {save_path5}")

except Exception as e:
    print(f"[ERROR] 최종 평가 실패: {e}")

print("\n>>> STEP 6 완료")

## STEP 7 | 결과 저장

In [ ]:
print("=== STEP 7 | 결과 저장 ===")

try:
    val_save_path = os.path.join(OUTPUT_DIR, "model_comparison_val.csv")
    df_val.to_csv(val_save_path, index=False, encoding="utf-8-sig")
    print(f"Val 비교표 저장 완료: {val_save_path}")
except Exception as e:
    print(f"[ERROR] Val CSV 저장 실패: {e}")

try:
    df_test_result = pd.DataFrame(results_test).T.reset_index()
    df_test_result.columns = ["모델명", "Accuracy", "Macro_F1", "AUC_ROC", "Class1_F1"]
    test_save_path = os.path.join(OUTPUT_DIR, "model_final_test.csv")
    df_test_result.to_csv(test_save_path, index=False, encoding="utf-8-sig")
    print(f"Test 최종 결과 저장 완료: {test_save_path}")
except Exception as e:
    print(f"[ERROR] Test CSV 저장 실패: {e}")

print("\n=== 전체 파이프라인 완료 ===")
print(f"최적 모델: {best_model_name} | Val AUC={best_auc:.3f}")
print("\n>>> STEP 7 완료")